# Libraries

In [1]:
!pip install -q -U llmcompressor datasets accelerate transformers huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.5/295.5 kB 6.6 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 67.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 563.6/563.6 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 38.0 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, w

In [2]:
import random
import numpy as np
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    set_seed,
)

from llmcompressor import oneshot
from llmcompressor.modifiers.awq import AWQModifier

from huggingface_hub import (
    login,
    create_repo,
    upload_folder,
)

from kaggle_secrets import UserSecretsClient
from datasets import load_dataset

2026-03-27 13:30:13.585550: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774618213.867654      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774618213.939948      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774618214.591055      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774618214.591117      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774618214.591123      55 computation_placer.cc:177] computation placer alr

# Global Variables

In [3]:
BASE_ID = "Qwen/Qwen3-4B"
MODEL_ID = "EdgeCompress01/Qwen3-4B-SparseGPT"
OUTPUT_DIR = "/kaggle/working/Qwen3-4B-SparseGPT-AWQ-4bit"
REPO_ID = f"EdgeCompress01/Qwen3-4B-SparseGPT-AWQ-4bit"
SEED = 42

# Functions

In [4]:
# --- 2. REPRODUCIBILITY ---
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

def format_chat(batch):
    texts = [
        tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        for messages in batch["messages"]
    ]
    return {"text": texts}

# Set Seed

In [5]:
set_reproducibility(SEED)

Global seed set to 42


# Huggingface Login

In [6]:
# --- 3. HUGGING FACE AUTHENTICATION ---
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logged in")

Logged in


# Load Model & Tokenizer

In [7]:
#Model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    subfolder="Models/70",
    device_map="auto",
    dtype=torch.bfloat16
)

#Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_ID)

config.json: 0.00B [00:00, ?B/s]

Models/70/model.safetensors:   0%|          | 0.00/8.04G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/213 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

# AWQ Quantization

**Configurations**

In [8]:
recipe = [
    AWQModifier(
        ignore=["lm_head", "embed_tokens"], 
        scheme="W4A16",
        targets=["Linear"]
    )
]

**Calibration Dataset**

In [9]:
dataset = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft").shuffle(seed=SEED).select(range(64))

formatted_dataset = dataset.map(
    format_chat,
    batched=True,
    remove_columns=dataset.column_names  # keep only "text"
)

README.md: 0.00B [00:00, ?B/s]

data/train_sft-00000-of-00003-a3ecf92756(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/train_sft-00001-of-00003-0a1804bcb6(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/train_sft-00002-of-00003-ee46ed25cf(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/test_sft-00000-of-00001-f7dfac4afe5(…):   0%|          | 0.00/81.2M [00:00<?, ?B/s]

...

Generating train_sft split:   0%|          | 0/207865 [00:00<?, ? examples/s]

Generating test_sft split:   0%|          | 0/23110 [00:00<?, ? examples/s]

Generating train_gen split:   0%|          | 0/256032 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/28304 [00:00<?, ? examples/s]

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

**Apply quantization**

In [10]:
oneshot(
    model=model,
    tokenizer=tokenizer,
    dataset=formatted_dataset,
    text_column="text",
    recipe=recipe,
    max_seq_length=512
)

2026-03-27T13:32:19.719560+0000 | __init__ | WARNING - Disabling tokenizer parallelism due to threading conflict between FastTokenizer and Datasets. Set TOKENIZERS_PARALLELISM=false to suppress this warning.


Tokenizing:   0%|          | 0/64 [00:00<?, ? examples/s]

2026-03-27T13:32:20.288633+0000 | _make_sampler | WARNING - Requested 512 samples but the provided dataset only has 64 samples.
2026-03-27T13:32:20.290310+0000 | reset | INFO - Compression lifecycle reset
2026-03-27T13:32:20.293918+0000 | from_modifiers | INFO - Creating recipe from modifiers
2026-03-27T13:32:20.386566+0000 | on_initialize | INFO - No AWQModifier.mappings provided, inferring from model...
2026-03-27T13:32:20.404433+0000 | _set_resolved_mappings | WARNING - skipping AWQ for model.layers.0.self_attn.v_proj for mapping AWQMapping(smooth_layer='re:.*v_proj$', balance_layers=['re:.*o_proj$'], activation_hook_target=None) because found incompatible balance layers
2026-03-27T13:32:20.406190+0000 | _set_resolved_mappings | WARNING - skipping AWQ for model.layers.1.self_attn.v_proj for mapping AWQMapping(smooth_layer='re:.*v_proj$', balance_layers=['re:.*o_proj$'], activation_hook_target=None) because found incompatible balance layers
2026-03-27T13:32:20.407386+0000 | _set_reso

W0327 13:32:20.537000 55 torch/fx/_symbolic_trace.py:52] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
Smoothing:  33%|███▎      | 1/3 [00:10<00:21, 10.65s/it]                                                             
Grid search for model.layers.0.post_attention_layernorm:   0%|          | 0/20 [00:00<?, ?it/s]
Grid search for model.layers.0.post_attention_layernorm:   0%|          | 0/20 [00:00<?, ?it/s, best_error=4.196e-05]
Grid search for model.layers.0.post_attention_layernorm:   5%|▌         | 1/20 [00:00<00:17,  1.09it/s, best_error=4.196e-05]
Grid search for model.layers.0.post_attention_layernorm:   5%|▌         | 1/20 [00:01<00:17,  1.09it/s, best_error=3.552e-05]
Grid search for model.layers.0.post_attention_layernorm:  10%|█         | 2/20 [00:01<00:16,  1.09it/s, best_error=3.552e-05]
Grid s

2026-03-27T13:56:27.048896+0000 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-03-27T13:56:27.050025+0000 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (post_attention_layer

**Saving**

In [11]:
model.config._name_or_path = ""

model.save_pretrained(OUTPUT_DIR,skip_sparsity_compression_stats=False)#VERY IMPORTANT
tokenizer.save_pretrained(OUTPUT_DIR)

print("Model saved successfully")

Calculating model sparsity: 100%|██████████| 903/903 [00:26<00:00, 34.28it/s]
Checking whether model follows 2:4 sparsity structure: 100%|██████████| 253/253 [00:25<00:00,  9.76it/s]
Compressing model: 252it [00:19, 12.91it/s]
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3970: UserWarning: Attempting to save a model with offloaded modules. Ensure that unallocated cpu memory exceeds the `shard_size` (5GB default)
  warnings.warn(


Saving checkpoint shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully


# PUSH TO HUGGING FACE

In [12]:
create_repo(REPO_ID, repo_type="model", exist_ok=True)

upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=REPO_ID,
    path_in_repo="Sparsity/70"
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/EdgeCompress01/Qwen3-4B-SparseGPT-AWQ-4bit/commit/6aefed1b665cc71b0f894fc6851a868df45127ec', commit_message='Upload folder using huggingface_hub', commit_description='', oid='6aefed1b665cc71b0f894fc6851a868df45127ec', pr_url=None, repo_url=RepoUrl('https://huggingface.co/EdgeCompress01/Qwen3-4B-SparseGPT-AWQ-4bit', endpoint='https://huggingface.co', repo_type='model', repo_id='EdgeCompress01/Qwen3-4B-SparseGPT-AWQ-4bit'), pr_revision=None, pr_num=None)